In [ ]:
from pathlib import Path
import openslide
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


# ============================================================
# 1. Utilidades básicas
# ============================================================

def ensure_uint8_rgb(img):
    """
    Asegura formato RGB uint8.
    """
    img = np.asarray(img)

    if img.ndim != 3 or img.shape[2] != 3:
        raise ValueError("La imagen debe tener forma H x W x 3")

    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    return img


def border_mask(shape, width=3):
    """
    Crea una máscara booleana del borde de la imagen.
    """
    h, w = shape
    mask = np.zeros((h, w), dtype=bool)

    mask[:width, :] = True
    mask[-width:, :] = True
    mask[:, :width] = True
    mask[:, -width:] = True

    return mask


def overlay_mask(img_rgb, mask, color=(255, 0, 0), alpha=0.6):
    """
    Pinta una máscara sobre la imagen.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    out = img_rgb.copy().astype(np.float32)
    mask = mask.astype(bool)
    color = np.array(color, dtype=np.float32)

    out[mask] = (1 - alpha) * out[mask] + alpha * color

    return np.clip(out, 0, 255).astype(np.uint8)


def overlay_contour(img_rgb, contour_mask, color=(255, 0, 0)):
    """
    Pinta un contorno sólido sobre la imagen.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    out = img_rgb.copy()
    contour_mask = contour_mask.astype(bool)

    out[contour_mask] = np.array(color, dtype=np.uint8)

    return out


def extract_ordered_points_from_mask(contour_mask):
    """
    Extrae puntos ordenados de una máscara de contorno.

    No intenta reconstruir topología perfecta.
    Ordena por X si la línea es más horizontal,
    o por Y si la línea es más vertical.
    """
    contour_mask = contour_mask.astype(bool)

    ys, xs = np.where(contour_mask)

    if len(xs) == 0:
        return None, None

    range_x = np.ptp(xs)
    range_y = np.ptp(ys)

    if range_x >= range_y:
        order = np.lexsort((ys, xs))
    else:
        order = np.lexsort((xs, ys))

    xs = xs[order]
    ys = ys[order]

    return xs, ys


def overlay_dotted_contour(
    img_rgb,
    contour_mask,
    dot_spacing=6,
    dot_radius=3,
    dot_color=(255, 255, 0),
):
    """
    Dibuja el contorno como puntos sobre la imagen.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    out = img_rgb.copy()

    xs, ys = extract_ordered_points_from_mask(contour_mask)

    if xs is None:
        return out, None, None

    for i in range(0, len(xs), dot_spacing):
        cv2.circle(
            out,
            (int(xs[i]), int(ys[i])),
            int(dot_radius),
            dot_color,
            thickness=-1
        )

    return out, xs, ys


# ============================================================
# 2. Detectar negro del padding / corte escaneado
# ============================================================

def detect_black_padding(
    img_rgb,
    black_v_threshold=35,
    black_mean_threshold=45,
    keep_only_border_connected=True,
    border_width=3,
):
    """
    Detecta el negro del padding/corte escaneado.

    Si keep_only_border_connected=True:
    solo mantiene componentes negras conectadas al borde.
    Esto evita coger puntos negros internos del tejido.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    _, _, V = cv2.split(hsv)

    mean_rgb = img_rgb.mean(axis=2)

    black_mask_raw = (
        (V <= black_v_threshold) &
        (mean_rgb <= black_mean_threshold)
    )

    if not keep_only_border_connected:
        return black_mask_raw

    h, w = black_mask_raw.shape
    bmask = border_mask((h, w), width=border_width)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        black_mask_raw.astype(np.uint8),
        connectivity=8
    )

    border_labels = np.unique(labels[black_mask_raw & bmask])
    border_labels = border_labels[border_labels != 0]

    black_padding_mask = np.isin(labels, border_labels)

    return black_padding_mask


# ============================================================
# 3. Detectar blancos candidatos a background
# ============================================================

def detect_white_background_candidates(
    img_rgb,
    white_v_threshold=190,
    white_s_threshold=50,
    white_dist_threshold=90,
):
    """
    Detecta píxeles blancos/casi blancos candidatos a background.

    Aquí pueden entrar blancos internos del espécimen.
    Después se selecciona el cluster blanco más cercano al negro.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    _, S, V = cv2.split(hsv)

    img_i16 = img_rgb.astype(np.int16)

    dist_to_white = np.sqrt(
        np.sum((255 - img_i16) ** 2, axis=2)
    )

    white_candidates = (
        (V >= white_v_threshold) &
        (S <= white_s_threshold) &
        (dist_to_white <= white_dist_threshold)
    )

    return white_candidates


# ============================================================
# 4. Elegir cluster blanco más cercano al negro
# ============================================================

def select_white_cluster_nearest_to_black(
    white_candidates,
    black_padding_mask,
    min_white_cluster_area_px=8000,
):
    """
    De todos los clusters blancos candidatos, se queda con el cluster
    cuya distancia mínima al background negro sea menor.
    """
    white_candidates = white_candidates.astype(bool)
    black_padding_mask = black_padding_mask.astype(bool)

    h, w = white_candidates.shape
    selected_white_bg_mask = np.zeros((h, w), dtype=bool)

    if black_padding_mask.sum() == 0:
        return selected_white_bg_mask, None, None, []

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        white_candidates.astype(np.uint8),
        connectivity=8
    )

    non_black = (~black_padding_mask).astype(np.uint8)
    dist_to_black = cv2.distanceTransform(non_black, cv2.DIST_L2, 5)

    all_components_info = []

    for lab in range(1, num_labels):
        area = int(stats[lab, cv2.CC_STAT_AREA])

        if area < min_white_cluster_area_px:
            continue

        component = labels == lab
        distances = dist_to_black[component]

        info = {
            "label": int(lab),
            "area": area,
            "min_dist_to_black": float(distances.min()),
            "mean_dist_to_black": float(distances.mean()),
        }

        all_components_info.append(info)

    if len(all_components_info) == 0:
        return selected_white_bg_mask, labels, None, []

    all_components_info = sorted(
        all_components_info,
        key=lambda x: (x["min_dist_to_black"], -x["area"])
    )

    selected_info = all_components_info[0]
    selected_label = selected_info["label"]

    selected_white_bg_mask = labels == selected_label

    return selected_white_bg_mask, labels, selected_info, all_components_info


# ============================================================
# 5. Utilidades para convertir máscara en línea abierta
# ============================================================

def morphological_skeleton(binary_mask):
    """
    Skeleton morfológico de una máscara binaria.
    Devuelve una línea de 1 píxel aproximadamente.
    """
    img = (binary_mask.astype(np.uint8) * 255).copy()
    skel = np.zeros_like(img)

    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))

    while True:
        eroded = cv2.erode(img, kernel)
        temp = cv2.dilate(eroded, kernel)
        temp = cv2.subtract(img, temp)
        skel = cv2.bitwise_or(skel, temp)
        img = eroded.copy()

        if cv2.countNonZero(img) == 0:
            break

    return skel > 0


def count_endpoints(component_mask):
    """
    Cuenta endpoints de una línea esquelética.
    Un endpoint es un píxel del skeleton con exactamente 1 vecino
    en conectividad 8.
    """
    comp = component_mask.astype(np.uint8)

    kernel = np.array([
        [1, 1, 1],
        [1, 10, 1],
        [1, 1, 1]
    ], dtype=np.uint8)

    conv = cv2.filter2D(comp, -1, kernel, borderType=cv2.BORDER_CONSTANT)

    endpoints = (comp == 1) & (conv == 11)

    return int(endpoints.sum())


def keep_only_open_line_components(
    skeleton_mask,
    min_component_length=15,
):
    """
    Mantiene solo componentes del skeleton que sean líneas abiertas.
    Elimina círculos cerrados y componentes pequeñas.
    """
    skeleton_mask = skeleton_mask.astype(bool)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        skeleton_mask.astype(np.uint8),
        connectivity=8
    )

    out = np.zeros_like(skeleton_mask, dtype=bool)

    for lab in range(1, num_labels):
        component = labels == lab
        area = int(stats[lab, cv2.CC_STAT_AREA])

        if area < min_component_length:
            continue

        n_endpoints = count_endpoints(component)

        if n_endpoints >= 1:
            out |= component

    return out


# ============================================================
# 6. Detectar franja negra uniforme en un lado de la tile
# ============================================================

def detect_uniform_black_side_strip(
    black_padding_mask,
    min_coverage_frac=0.95,
    max_thickness_std=1.5,
    max_thickness_range=4,
    min_mean_thickness=1,
    max_mean_thickness=15,   # NUEVO: grosor medio máximo permitido
):
    """
    Detecta si existe una franja negra uniforme y fina ocupando uno de los 4 lados
    de la imagen (top, bottom, left, right).

    Condición:
    - la franja debe recorrer prácticamente todo el lado
    - su grosor debe ser aproximadamente constante
    - su grosor medio debe estar entre min_mean_thickness y max_mean_thickness

    Devuelve:
    - found: bool
    - side: 'top' | 'bottom' | 'left' | 'right' | None
    - thicknesses: vector de espesores por columna/fila
    """

    black = black_padding_mask.astype(bool)
    h, w = black.shape

    def evaluate_side(thicknesses):
        present = thicknesses > 0
        coverage_frac = present.mean()

        if coverage_frac < min_coverage_frac:
            return False

        t = thicknesses[present]
        if len(t) == 0:
            return False

        mean_t = float(np.mean(t))
        std_t = float(np.std(t))
        range_t = int(np.max(t) - np.min(t))

        # Grosor mínimo
        if mean_t < min_mean_thickness:
            return False

        # NUEVO: grosor máximo
        if mean_t > max_mean_thickness:
            return False

        # Uniformidad por desviación estándar
        if std_t > max_thickness_std:
            return False

        # Uniformidad por rango máximo-mínimo
        if range_t > max_thickness_range:
            return False

        return True

    # --------------------------------------------------------
    # TOP: grosor negro desde arriba hacia abajo, por columna
    # --------------------------------------------------------
    top_thickness = np.zeros(w, dtype=int)
    for x in range(w):
        t = 0
        while t < h and black[t, x]:
            t += 1
        top_thickness[x] = t

    if evaluate_side(top_thickness):
        return True, "top", top_thickness

    # --------------------------------------------------------
    # BOTTOM: grosor negro desde abajo hacia arriba, por columna
    # --------------------------------------------------------
    bottom_thickness = np.zeros(w, dtype=int)
    for x in range(w):
        t = 0
        y = h - 1
        while y - t >= 0 and black[y - t, x]:
            t += 1
        bottom_thickness[x] = t

    if evaluate_side(bottom_thickness):
        return True, "bottom", bottom_thickness

    # --------------------------------------------------------
    # LEFT: grosor negro desde izquierda hacia derecha, por fila
    # --------------------------------------------------------
    left_thickness = np.zeros(h, dtype=int)
    for y in range(h):
        t = 0
        while t < w and black[y, t]:
            t += 1
        left_thickness[y] = t

    if evaluate_side(left_thickness):
        return True, "left", left_thickness

    # --------------------------------------------------------
    # RIGHT: grosor negro desde derecha hacia izquierda, por fila
    # --------------------------------------------------------
    right_thickness = np.zeros(h, dtype=int)
    for y in range(h):
        t = 0
        x = w - 1
        while x - t >= 0 and black[y, x - t]:
            t += 1
        right_thickness[y] = t

    if evaluate_side(right_thickness):
        return True, "right", right_thickness

    return False, None, None


# ============================================================
# 7. Sacar contorno desde el background final
# ============================================================

def extract_contour_from_final_background(
    selected_white_bg_mask,
    black_padding_mask,
    contour_thickness=2,
    force_open_line=True,
    min_component_length=15,

    # compatibilidad con tu llamada actual
    img_rgb=None,
    white_candidates=None,
    use_black_touching_specimen_fallback=True,
    min_black_touching_line_length=30,

    # parámetros del nuevo caso
    use_uniform_side_black_strip_rule=True,
    min_side_strip_coverage_frac=0.95,
    max_side_strip_thickness_std=1.5,
    max_side_strip_thickness_range=4,
):
    """
    Saca el contorno entre background y espécimen.

    NUEVA REGLA PRIORITARIA:
    Si existe una franja negra uniforme ocupando un lado completo
    de la imagen, el contorno se fuerza como la línea interior
    de esa franja negra.

    Si no, sigue con el caso normal:
    contorno entre background blanco final y espécimen.
    """

    bg_mask = selected_white_bg_mask.astype(bool)
    black_mask = black_padding_mask.astype(bool)

    if black_mask.sum() == 0:
        contour_mask = np.zeros_like(bg_mask, dtype=bool)
        return contour_mask, False, "no se encontró contorno"

    h, w = black_mask.shape

    # ========================================================
    # NUEVA REGLA:
    # franja negra uniforme en uno de los lados
    # ========================================================
    if use_uniform_side_black_strip_rule:
        found_strip, strip_side, strip_thicknesses = detect_uniform_black_side_strip(
            black_mask,
            min_coverage_frac=min_side_strip_coverage_frac,
            max_thickness_std=max_side_strip_thickness_std,
            max_thickness_range=max_side_strip_thickness_range,
            min_mean_thickness=1,
            max_mean_thickness=15,   # NUEVO
        )

        if found_strip:
            contour_line = np.zeros_like(black_mask, dtype=bool)

            if strip_side == "top":
                for x in range(w):
                    t = strip_thicknesses[x]
                    if t > 0 and t < h:
                        contour_line[t, x] = True

            elif strip_side == "bottom":
                for x in range(w):
                    t = strip_thicknesses[x]
                    if t > 0 and t < h:
                        y = h - t - 1
                        if 0 <= y < h:
                            contour_line[y, x] = True

            elif strip_side == "left":
                for y in range(h):
                    t = strip_thicknesses[y]
                    if t > 0 and t < w:
                        contour_line[y, t] = True

            elif strip_side == "right":
                for y in range(h):
                    t = strip_thicknesses[y]
                    if t > 0 and t < w:
                        x = w - t - 1
                        if 0 <= x < w:
                            contour_line[y, x] = True

            # adelgazar / limpiar si quieres mantener el mismo comportamiento
            if contour_line.sum() > 0 and force_open_line:
                contour_line = morphological_skeleton(contour_line)

                contour_line = keep_only_open_line_components(
                    contour_line,
                    min_component_length=min_component_length
                )

            if contour_line.sum() > 0:
                return contour_line, True, f"contorno detectado usando franja negra uniforme en {strip_side}"

    # ========================================================
    # CASO NORMAL:
    # contorno entre background blanco final y espécimen
    # ========================================================
    if bg_mask.sum() == 0:
        contour_mask = np.zeros_like(bg_mask, dtype=bool)
        return contour_mask, False, "no se encontró contorno"

    specimen_mask = ~(bg_mask | black_mask)

    ksize = max(3, 2 * contour_thickness + 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))

    bg_dilated = cv2.dilate(bg_mask.astype(np.uint8), kernel) > 0

    # Solo píxeles del espécimen tocando background blanco
    contour_mask = bg_dilated & specimen_mask

    if contour_mask.sum() == 0:
        contour_mask = np.zeros_like(bg_mask, dtype=bool)
        return contour_mask, False, "no se encontró contorno"

    if force_open_line:
        contour_skeleton = morphological_skeleton(contour_mask)

        contour_line = keep_only_open_line_components(
            contour_skeleton,
            min_component_length=min_component_length
        )

        if contour_line.sum() == 0:
            contour_line = np.zeros_like(bg_mask, dtype=bool)
            return contour_line, False, "no se encontró contorno"

        return contour_line, True, "contorno detectado"

    return contour_mask, True, "contorno detectado"


# ============================================================
# 8. Pipeline negro -> blanco -> contorno
# ============================================================

def pipeline_background_from_black_then_white(
    img_rgb,

    # Negro
    black_v_threshold=35,
    black_mean_threshold=45,
    black_border_width=3,

    # Blanco candidato
    white_v_threshold=190,
    white_s_threshold=50,
    white_dist_threshold=90,

    # Cluster blanco
    min_white_cluster_area_px=8000,

    # Contorno
    contour_thickness=2,
    force_open_line=True,
    min_component_length=15,

    # Franja negra uniforme
    use_uniform_side_black_strip_rule=True,
    min_side_strip_coverage_frac=0.95,
    max_side_strip_thickness_std=1.5,
    max_side_strip_thickness_range=4,
):
    """
    Pipeline:
    1. Detecta negro del padding/corte.
    2. Detecta blancos candidatos.
    3. Selecciona cluster blanco más cercano al negro.
    4. Extrae contorno final.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    black_padding_mask = detect_black_padding(
        img_rgb,
        black_v_threshold=black_v_threshold,
        black_mean_threshold=black_mean_threshold,
        keep_only_border_connected=True,
        border_width=black_border_width,
    )

    white_candidates = detect_white_background_candidates(
        img_rgb,
        white_v_threshold=white_v_threshold,
        white_s_threshold=white_s_threshold,
        white_dist_threshold=white_dist_threshold,
    )

    selected_white_bg_mask, white_labels, selected_info, all_components_info = (
        select_white_cluster_nearest_to_black(
            white_candidates=white_candidates,
            black_padding_mask=black_padding_mask,
            min_white_cluster_area_px=min_white_cluster_area_px,
        )
    )

    contour_mask, has_contour, contour_message = extract_contour_from_final_background(
        selected_white_bg_mask=selected_white_bg_mask,
        black_padding_mask=black_padding_mask,
        contour_thickness=contour_thickness,
        force_open_line=force_open_line,
        min_component_length=min_component_length,

        use_uniform_side_black_strip_rule=use_uniform_side_black_strip_rule,
        min_side_strip_coverage_frac=min_side_strip_coverage_frac,
        max_side_strip_thickness_std=max_side_strip_thickness_std,
        max_side_strip_thickness_range=max_side_strip_thickness_range,
    )

    result = {
        "black_padding_mask": black_padding_mask,
        "white_candidates": white_candidates,
        "selected_white_bg_mask": selected_white_bg_mask,
        "white_labels": white_labels,
        "selected_info": selected_info,
        "all_components_info": all_components_info,

        "contour_mask": contour_mask,
        "has_contour": has_contour,
        "contour_message": contour_message,
    }

    return result


# ============================================================
# 9. Visualización de debug para una tile
# ============================================================

def show_tile_background_debug(
    tile_dict,
    result,
    figsize=(28, 7),
    show_dotted=True,
    dot_spacing=6,
    dot_radius=3,
    dot_color=(255, 255, 0),
):
    """
    Muestra:
    1. original
    2. negro en rojo
    3. blancos candidatos en verde
    4. cluster blanco final en amarillo
    5. contorno final
    """
    img_rgb = ensure_uint8_rgb(tile_dict["img"])

    black_overlay = overlay_mask(
        img_rgb,
        result["black_padding_mask"],
        color=(255, 0, 0),
        alpha=0.75
    )

    white_overlay = overlay_mask(
        img_rgb,
        result["white_candidates"],
        color=(0, 255, 0),
        alpha=0.55
    )

    final_overlay = overlay_mask(
        img_rgb,
        result["selected_white_bg_mask"],
        color=(255, 255, 0),
        alpha=0.70
    )

    fig, axes = plt.subplots(1, 5, figsize=figsize)

    tile_title = (
        f"Tile {tile_dict['id']} | row={tile_dict['row']}, col={tile_dict['col']}\n"
        f"x={tile_dict['x0']}, y={tile_dict['y0']}, "
        f"w={tile_dict['w']}, h={tile_dict['h']}"
    )

    axes[0].imshow(img_rgb)
    axes[0].set_title(tile_title + "\nOriginal")
    axes[0].axis("off")

    axes[1].imshow(black_overlay)
    axes[1].set_title("Negro padding/corte\nen rojo")
    axes[1].axis("off")

    axes[2].imshow(white_overlay)
    axes[2].set_title("Blancos candidatos\nen verde")
    axes[2].axis("off")

    axes[3].imshow(final_overlay)
    axes[3].set_title("Cluster blanco final\nen amarillo")
    axes[3].axis("off")

    if result["has_contour"]:
        if show_dotted:
            contour_overlay, _, _ = overlay_dotted_contour(
                img_rgb,
                result["contour_mask"],
                dot_spacing=dot_spacing,
                dot_radius=dot_radius,
                dot_color=dot_color,
            )
            axes[4].imshow(contour_overlay)
            axes[4].set_title(
                "Contorno final punteado\n"
                f"{result['contour_message']}"
            )
        else:
            contour_overlay = overlay_contour(
                img_rgb,
                result["contour_mask"],
                color=(255, 0, 0)
            )
            axes[4].imshow(contour_overlay)
            axes[4].set_title(
                "Contorno final sólido\n"
                f"{result['contour_message']}"
            )

        axes[4].axis("off")

    else:
        axes[4].imshow(img_rgb)
        axes[4].set_title("Sin contorno")
        axes[4].axis("off")
        axes[4].text(
            0.5, 0.5,
            result["contour_message"],
            color="red",
            fontsize=12,
            ha="center",
            va="center",
            transform=axes[4].transAxes,
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="red")
        )

    plt.tight_layout()
    plt.show()


def show_tile_background_debug_multiple(
    tiles,
    figsize_per_row=6,
    show_dotted=True,
    dot_spacing=6,
    dot_radius=3,
    dot_color=(255, 255, 0),
):
    """
    Muestra varias tiles procesadas.
    Cada fila = una tile.
    """
    n = len(tiles)

    if n == 0:
        print("No hay tiles para mostrar.")
        return

    fig, axes = plt.subplots(n, 5, figsize=(30, figsize_per_row * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, t in enumerate(tiles):
        img_rgb = ensure_uint8_rgb(t["img"])
        result = t["background_pipeline_result"]

        black_overlay = overlay_mask(
            img_rgb,
            result["black_padding_mask"],
            color=(255, 0, 0),
            alpha=0.75
        )

        white_overlay = overlay_mask(
            img_rgb,
            result["white_candidates"],
            color=(0, 255, 0),
            alpha=0.55
        )

        final_overlay = overlay_mask(
            img_rgb,
            result["selected_white_bg_mask"],
            color=(255, 255, 0),
            alpha=0.70
        )

        row_title = (
            f"Tile {t['id']} | row={t['row']}, col={t['col']}\n"
            f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']}"
        )

        axes[i, 0].imshow(img_rgb)
        axes[i, 0].set_title(row_title + "\nOriginal")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(black_overlay)
        axes[i, 1].set_title("Negro padding/corte\nen rojo")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(white_overlay)
        axes[i, 2].set_title("Blancos candidatos\nen verde")
        axes[i, 2].axis("off")

        axes[i, 3].imshow(final_overlay)
        axes[i, 3].set_title("Cluster blanco final\nen amarillo")
        axes[i, 3].axis("off")

        if result["has_contour"]:
            if show_dotted:
                contour_overlay, _, _ = overlay_dotted_contour(
                    img_rgb,
                    result["contour_mask"],
                    dot_spacing=dot_spacing,
                    dot_radius=dot_radius,
                    dot_color=dot_color,
                )
            else:
                contour_overlay = overlay_contour(
                    img_rgb,
                    result["contour_mask"],
                    color=(255, 0, 0)
                )

            axes[i, 4].imshow(contour_overlay)
            axes[i, 4].set_title(
                "Contorno final\n"
                f"{result['contour_message']}"
            )
            axes[i, 4].axis("off")

        else:
            axes[i, 4].imshow(img_rgb)
            axes[i, 4].set_title("Sin contorno")
            axes[i, 4].axis("off")
            axes[i, 4].text(
                0.5, 0.5,
                result["contour_message"],
                color="red",
                fontsize=12,
                ha="center",
                va="center",
                transform=axes[i, 4].transAxes,
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="red")
            )

    plt.tight_layout()
    plt.show()


def print_background_pipeline_summary(result):
    """
    Imprime resumen del pipeline de una tile.
    """
    print("BLACK PIXELS:", int(result["black_padding_mask"].sum()))
    print("WHITE CANDIDATE PIXELS:", int(result["white_candidates"].sum()))
    print("FINAL WHITE BACKGROUND PIXELS:", int(result["selected_white_bg_mask"].sum()))
    print("HAS_CONTOUR:", result["has_contour"])
    print("CONTOUR MESSAGE:", result["contour_message"])

    print("CLUSTER SELECCIONADO:")
    if result["selected_info"] is None:
        print("No se seleccionó ningún cluster blanco.")
    else:
        print(result["selected_info"])


# ============================================================
# 10. Función principal combinada
# ============================================================

def select_level_tiles_and_detect_background_contour(
    slide_path,
    target_level=4,
    tile_w=800,
    tile_h=1400,

    # =========================================================
    # Parámetros detección contorno global en thumbnail
    # =========================================================
    thumb_sat_threshold=15,
    thumb_val_threshold=220,
    thumb_dist_threshold=20,
    thumb_kernel_open_size=5,
    thumb_kernel_close_size=9,
    thumb_kernel_smooth_size=7,
    contour_band_thickness_thumb=10,

    # =========================================================
    # Parámetros filtro de tiles en target_level
    # =========================================================
    specimen_frac_threshold=0.02,
    dist_threshold=18,
    sat_threshold=18,
    value_threshold=220,
    min_contour_overlap_pixels=5,
    show_removed=True,

    # =========================================================
    # Parámetros pipeline negro/blanco/contorno por tile
    # =========================================================
    black_v_threshold=35,
    black_mean_threshold=45,
    black_border_width=3,

    white_v_threshold=190,
    white_s_threshold=50,
    white_dist_threshold=90,

    min_white_cluster_area_px=8000,

    contour_thickness=2,
    force_open_line=True,
    min_component_length=15,

    use_uniform_side_black_strip_rule=True,
    min_side_strip_coverage_frac=0.95,
    max_side_strip_thickness_std=1.5,
    max_side_strip_thickness_range=4,

    # =========================================================
    # Parámetros visuales del contorno punteado
    # =========================================================
    dot_spacing=6,
    dot_radius=3,
    dot_color=(255, 255, 0),

    # =========================================================
    # Visualización
    # =========================================================
    show_global_contour=True,
    show_all_grid=True,
    show_selected_grid=True,
    show_tile_contours=True,
    show_tile_contour_steps=True,
    show_tile_batches=False,
    batch_size=3,
):
    """
    Pipeline combinado:

    1. Abre la WSI con OpenSlide.
    2. Detecta el contorno externo aproximado del espécimen en la thumbnail.
    3. Genera una banda alrededor del contorno global.
    4. Lee el target_level completo.
    5. Segmenta el espécimen en target_level.
    6. Genera tiles en target_level.
    7. Conserva solo tiles que:
       - no son negras,
       - contienen suficiente espécimen,
       - intersectan la banda del contorno global.
    8. Para cada tile conservada:
       - detecta negro del padding/corte,
       - detecta blancos candidatos,
       - elige cluster blanco más cercano al negro,
       - extrae el contorno final,
       - aplica la regla especial de franja negra uniforme,
       - guarda overlay punteado y máscaras dentro del diccionario de la tile.
    """

    # =========================================================
    # 0. Abrir slide
    # =========================================================
    slide_path = Path(slide_path)
    slide = openslide.OpenSlide(str(slide_path))

    print("Level dimensions:", slide.level_dimensions)
    print("Level downsamples:", slide.level_downsamples)

    if target_level < 0 or target_level >= len(slide.level_dimensions):
        raise ValueError(f"target_level={target_level} no existe en esta WSI.")

    lvl_w, lvl_h = slide.level_dimensions[target_level]

    print(f"\nAnalizando level {target_level}: {lvl_w} x {lvl_h} px")

    # =========================================================
    # 1. Leer thumbnail
    # =========================================================
    if "thumbnail" in slide.associated_images:
        thumb = slide.associated_images["thumbnail"].convert("RGB")
    else:
        thumb = slide.get_thumbnail((1200, 1200)).convert("RGB")

    thumb_np = np.array(thumb)
    thumb_h, thumb_w = thumb_np.shape[:2]

    # =========================================================
    # 2. Detección del contorno global en thumbnail
    # =========================================================
    img_thumb = thumb_np.copy()

    hsv_thumb = cv2.cvtColor(img_thumb, cv2.COLOR_RGB2HSV)
    _, S_thumb, V_thumb = cv2.split(hsv_thumb)

    border_pixels = np.concatenate([
        img_thumb[0, :, :],
        img_thumb[-1, :, :],
        img_thumb[:, 0, :],
        img_thumb[:, -1, :]
    ], axis=0).astype(np.float32)

    bg_color_thumb = np.median(border_pixels, axis=0)

    dist_thumb = np.linalg.norm(
        img_thumb.astype(np.float32) - bg_color_thumb[None, None, :],
        axis=2
    )

    mask_hsv_thumb = (
        (S_thumb > thumb_sat_threshold) |
        (V_thumb < thumb_val_threshold)
    ).astype(np.uint8) * 255

    mask_dist_thumb = (dist_thumb > thumb_dist_threshold).astype(np.uint8) * 255

    mask_tissue_thumb = np.maximum(mask_hsv_thumb, mask_dist_thumb)

    kernel_open_thumb = np.ones(
        (thumb_kernel_open_size, thumb_kernel_open_size),
        np.uint8
    )

    kernel_close_thumb = np.ones(
        (thumb_kernel_close_size, thumb_kernel_close_size),
        np.uint8
    )

    mask_tissue_thumb = cv2.morphologyEx(
        mask_tissue_thumb,
        cv2.MORPH_OPEN,
        kernel_open_thumb
    )

    mask_tissue_thumb = cv2.morphologyEx(
        mask_tissue_thumb,
        cv2.MORPH_CLOSE,
        kernel_close_thumb
    )

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_tissue_thumb,
        connectivity=8
    )

    if nlab <= 1:
        raise ValueError("No se ha detectado espécimen en la thumbnail.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask_thumb_main = np.uint8(labels == largest) * 255

    # Rellenar huecos internos
    h_t, w_t = mask_thumb_main.shape
    flood = mask_thumb_main.copy()
    flood_mask = np.zeros((h_t + 2, w_t + 2), np.uint8)

    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    mask_thumb_filled = cv2.bitwise_or(mask_thumb_main, holes)

    # Suavizado del borde
    kernel_smooth_thumb = np.ones(
        (thumb_kernel_smooth_size, thumb_kernel_smooth_size),
        np.uint8
    )

    mask_thumb_smooth = cv2.morphologyEx(
        mask_thumb_filled,
        cv2.MORPH_CLOSE,
        kernel_smooth_thumb
    )

    mask_thumb_smooth = cv2.morphologyEx(
        mask_thumb_smooth,
        cv2.MORPH_OPEN,
        kernel_smooth_thumb
    )

    contours_thumb, _ = cv2.findContours(
        mask_thumb_smooth,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    if len(contours_thumb) == 0:
        raise ValueError("No se ha encontrado contorno externo en thumbnail.")

    coarse_contour_thumb = max(contours_thumb, key=cv2.contourArea)

    contour_band_thumb = np.zeros_like(mask_thumb_smooth)

    cv2.drawContours(
        contour_band_thumb,
        [coarse_contour_thumb],
        -1,
        255,
        thickness=contour_band_thickness_thumb
    )

    mask_thumb_filled = mask_thumb_smooth

    # =========================================================
    # 3. Mostrar contorno global
    # =========================================================
    if show_global_contour:
        overlay_thumb_contour = thumb_np.copy()

        cv2.drawContours(
            overlay_thumb_contour,
            [coarse_contour_thumb],
            -1,
            (255, 255, 0),
            2
        )

        plt.figure(figsize=(8, 14))
        plt.imshow(overlay_thumb_contour)
        plt.axis("off")
        plt.title("Contorno global detectado sobre thumbnail")
        plt.show()

    # =========================================================
    # 4. Leer imagen completa del target_level
    # =========================================================
    img_level = slide.read_region(
        (0, 0),
        target_level,
        (lvl_w, lvl_h)
    ).convert("RGB")

    img_level = np.array(img_level)

    img_float = img_level.astype(np.float32)

    border_pixels_level = np.concatenate([
        img_float[0, :, :],
        img_float[-1, :, :],
        img_float[:, 0, :],
        img_float[:, -1, :]
    ], axis=0)

    bg_color_level = np.median(border_pixels_level, axis=0)

    dist_level = np.linalg.norm(
        img_float - bg_color_level[None, None, :],
        axis=2
    )

    hsv_level = cv2.cvtColor(img_level, cv2.COLOR_RGB2HSV)
    _, S_level, V_level = cv2.split(hsv_level)

    mask_level = (
        (dist_level > dist_threshold) |
        (S_level > sat_threshold) |
        (V_level < value_threshold)
    ).astype(np.uint8) * 255

    kernel_level = np.ones((5, 5), np.uint8)

    mask_level = cv2.morphologyEx(
        mask_level,
        cv2.MORPH_OPEN,
        kernel_level
    )

    mask_level = cv2.morphologyEx(
        mask_level,
        cv2.MORPH_CLOSE,
        kernel_level
    )

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_level,
        connectivity=8
    )

    if nlab <= 1:
        raise ValueError("No se ha detectado espécimen en el target_level.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask_main = np.uint8(labels == largest) * 255

    # Rellenar huecos internos
    h, w = mask_main.shape
    flood = mask_main.copy()
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)

    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    mask_filled = cv2.bitwise_or(mask_main, holes)

    # =========================================================
    # 5. Construir tiles en target_level
    # =========================================================
    tiles = []
    tile_id = 1

    scale_x = thumb_w / lvl_w
    scale_y = thumb_h / lvl_h

    row_id = 0

    for y0 in range(0, lvl_h, tile_h):
        col_id = 0

        for x0 in range(0, lvl_w, tile_w):
            w_tile = min(tile_w, lvl_w - x0)
            h_tile = min(tile_h, lvl_h - y0)

            tile_img = img_level[y0:y0 + h_tile, x0:x0 + w_tile].copy()
            tile_mask = mask_filled[y0:y0 + h_tile, x0:x0 + w_tile]

            is_black = bool(np.all(tile_img == 0))
            specimen_frac = tile_mask.mean() / 255.0

            # Proyección del tile sobre thumbnail
            x0_t = int(np.floor(x0 * scale_x))
            y0_t = int(np.floor(y0 * scale_y))
            x1_t = int(np.ceil((x0 + w_tile) * scale_x))
            y1_t = int(np.ceil((y0 + h_tile) * scale_y))

            x0_t = max(0, min(x0_t, thumb_w))
            x1_t = max(0, min(x1_t, thumb_w))
            y0_t = max(0, min(y0_t, thumb_h))
            y1_t = max(0, min(y1_t, thumb_h))

            if x1_t <= x0_t or y1_t <= y0_t:
                contour_overlap_pixels = 0
            else:
                contour_overlap_pixels = int(
                    np.count_nonzero(
                        contour_band_thumb[y0_t:y1_t, x0_t:x1_t]
                    )
                )

            keep = (
                (not is_black)
                and (specimen_frac > specimen_frac_threshold)
                and (contour_overlap_pixels >= min_contour_overlap_pixels)
            )

            reason = []

            if is_black:
                reason.append("100% negra")

            if specimen_frac <= specimen_frac_threshold:
                reason.append(f"sin espécimen suficiente ({specimen_frac:.3f})")

            if contour_overlap_pixels < min_contour_overlap_pixels:
                reason.append(
                    f"sin solape suficiente con contorno "
                    f"({contour_overlap_pixels})"
                )

            reason_text = "conservar" if keep else ", ".join(reason)

            tiles.append({
                "id": tile_id,
                "row": row_id,
                "col": col_id,
                "x0": x0,
                "y0": y0,
                "w": w_tile,
                "h": h_tile,
                "img": tile_img,
                "is_black": is_black,
                "specimen_frac": specimen_frac,
                "contour_overlap_pixels": contour_overlap_pixels,
                "keep": keep,
                "reason": reason_text,
            })

            tile_id += 1
            col_id += 1

        row_id += 1

    kept_tiles = [t for t in tiles if t["keep"]]
    removed_tiles = [t for t in tiles if not t["keep"]]

    # =========================================================
    # 6. Overview: todas las cuadrículas
    # =========================================================
    if show_all_grid:
        fig, ax = plt.subplots(figsize=(8, 14))
        ax.imshow(thumb_np)

        cv2_contour = coarse_contour_thumb[:, 0, :]

        ax.plot(
            cv2_contour[:, 0],
            cv2_contour[:, 1],
            color="yellow",
            linewidth=1.5
        )

        for t in tiles:
            x_t = t["x0"] * scale_x
            y_t = t["y0"] * scale_y
            w_t = t["w"] * scale_x
            h_t = t["h"] * scale_y

            rect = Rectangle(
                (x_t, y_t),
                w_t,
                h_t,
                fill=False,
                edgecolor="cyan",
                linewidth=0.8
            )
            ax.add_patch(rect)

            ax.text(
                x_t + 2,
                y_t + 12,
                str(t["id"]),
                color="yellow",
                fontsize=7,
                bbox=dict(
                    facecolor="black",
                    alpha=0.6,
                    edgecolor="none",
                    pad=1
                )
            )

        ax.set_title(f"Todas las cuadrículas - level {target_level} + contorno global")
        ax.axis("off")
        plt.show()

    # =========================================================
    # 7. Overview final: tiles seleccionadas
    # =========================================================
    if show_selected_grid:
        fig, ax = plt.subplots(figsize=(8, 14))
        ax.imshow(thumb_np)

        ax.contour(
            contour_band_thumb > 0,
            levels=[0.5],
            colors="yellow",
            linewidths=1
        )

        for t in tiles:
            x_t = t["x0"] * scale_x
            y_t = t["y0"] * scale_y
            w_t = t["w"] * scale_x
            h_t = t["h"] * scale_y

            if not t["keep"]:
                if not show_removed:
                    continue

                color = "red"
                linestyle = "--"
            else:
                color = "lime"
                linestyle = "-"

            rect = Rectangle(
                (x_t, y_t),
                w_t,
                h_t,
                fill=False,
                edgecolor=color,
                linewidth=1.2,
                linestyle=linestyle
            )
            ax.add_patch(rect)

            ax.text(
                x_t + 2,
                y_t + 12,
                str(t["id"]),
                color=color,
                fontsize=7,
                bbox=dict(
                    facecolor="black",
                    alpha=0.6,
                    edgecolor="none",
                    pad=1
                )
            )

        ax.set_title(
            f"Tiles seleccionadas por contorno global - level {target_level}\n"
            f"Verde = conservar | Rojo = eliminar"
        )
        ax.axis("off")
        plt.show()

    # =========================================================
    # 8. Texto tiles eliminadas / conservadas
    # =========================================================
    print("\n==============================")
    print("TILES ELIMINADAS")
    print("==============================")

    if len(removed_tiles) == 0:
        print("Ninguna")
    else:
        for t in removed_tiles:
            print(
                f"Tile {t['id']:02d} | "
                f"row={t['row']}, col={t['col']} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
                f"motivo: {t['reason']}"
            )

    print("\n==============================")
    print("TILES CONSERVADAS POR CONTORNO GLOBAL")
    print("==============================")

    if len(kept_tiles) == 0:
        print("Ninguna")
    else:
        for t in kept_tiles:
            print(
                f"Tile {t['id']:02d} | "
                f"row={t['row']}, col={t['col']} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
                f"specimen_frac={t['specimen_frac']:.3f} | "
                f"contour_overlap_pixels={t['contour_overlap_pixels']}"
            )

    # =========================================================
    # 9. Aplicar pipeline negro/blanco/contorno a cada tile
    # =========================================================
    print("\n==============================")
    print("DETECTANDO CONTORNO FINAL EN TILES CONSERVADAS")
    print("==============================")

    for t in kept_tiles:
        tile_rgb = t["img"].copy()

        result = pipeline_background_from_black_then_white(
            tile_rgb,

            black_v_threshold=black_v_threshold,
            black_mean_threshold=black_mean_threshold,
            black_border_width=black_border_width,

            white_v_threshold=white_v_threshold,
            white_s_threshold=white_s_threshold,
            white_dist_threshold=white_dist_threshold,

            min_white_cluster_area_px=min_white_cluster_area_px,

            contour_thickness=contour_thickness,
            force_open_line=force_open_line,
            min_component_length=min_component_length,

            use_uniform_side_black_strip_rule=use_uniform_side_black_strip_rule,
            min_side_strip_coverage_frac=min_side_strip_coverage_frac,
            max_side_strip_thickness_std=max_side_strip_thickness_std,
            max_side_strip_thickness_range=max_side_strip_thickness_range,
        )

        contour_mask = result["contour_mask"]

        overlay_punteado, xs_contorno, ys_contorno = overlay_dotted_contour(
            tile_rgb,
            contour_mask,
            dot_spacing=dot_spacing,
            dot_radius=dot_radius,
            dot_color=dot_color,
        )

        overlay_solido = overlay_contour(
            tile_rgb,
            contour_mask,
            color=(255, 0, 0)
        )

        # Guardar resultado completo dentro de la tile
        t["background_pipeline_result"] = result

        t["black_padding_mask"] = result["black_padding_mask"]
        t["white_candidates"] = result["white_candidates"]
        t["selected_white_bg_mask"] = result["selected_white_bg_mask"]

        t["contour_detected"] = result["has_contour"]
        t["contour_error"] = None if result["has_contour"] else result["contour_message"]
        t["contour_message"] = result["contour_message"]

        t["mask_contorno_tile"] = (contour_mask.astype(np.uint8) * 255)
        t["contour_mask"] = contour_mask

        t["overlay_contorno_punteado"] = overlay_punteado
        t["overlay_contorno_solido"] = overlay_solido

        t["xs_contorno"] = xs_contorno
        t["ys_contorno"] = ys_contorno

        print("\n--------------------------------")
        print(
            f"Tile {t['id']:02d} | "
            f"row={t['row']}, col={t['col']} | "
            f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']}"
        )
        print_background_pipeline_summary(result)

        if result["has_contour"]:
            n_points = int(contour_mask.sum())
            print(f"Píxeles de contorno: {n_points}")
        else:
            print("No se detectó contorno final.")

        if show_tile_contours and show_tile_contour_steps:
            show_tile_background_debug(
                t,
                result,
                figsize=(28, 7),
                show_dotted=True,
                dot_spacing=dot_spacing,
                dot_radius=dot_radius,
                dot_color=dot_color,
            )

        elif show_tile_contours and not show_tile_contour_steps:
            aspect = t["h"] / t["w"]
            fig_w = 10
            fig_h = min(16, max(6, fig_w * aspect))

            plt.figure(figsize=(fig_w, fig_h))
            plt.imshow(t["overlay_contorno_punteado"])
            plt.axis("off")
            plt.title(
                f"Tile {t['id']} - contorno final punteado\n"
                f"row={t['row']}, col={t['col']} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']}\n"
                f"{t['contour_message']}"
            )
            plt.show()

    # =========================================================
    # 10. Visualización opcional por bloques
    # =========================================================
    if show_tile_batches:
        for start in range(0, len(kept_tiles), batch_size):
            end = start + batch_size
            batch = kept_tiles[start:end]

            print(
                f"\nMostrando tiles conservadas "
                f"{start + 1} a {min(end, len(kept_tiles))} "
                f"de {len(kept_tiles)}"
            )

            show_tile_background_debug_multiple(
                batch,
                figsize_per_row=6,
                show_dotted=True,
                dot_spacing=dot_spacing,
                dot_radius=dot_radius,
                dot_color=dot_color,
            )

    # =========================================================
    # 11. Resumen final
    # =========================================================
    detected = [
        t for t in kept_tiles
        if t.get("contour_detected", False)
    ]

    not_detected = [
        t for t in kept_tiles
        if not t.get("contour_detected", False)
    ]

    print("\n==============================")
    print("RESUMEN FINAL CONTORNO")
    print("==============================")
    print(f"Tiles conservadas: {len(kept_tiles)}")
    print(f"Tiles con contorno detectado: {len(detected)}")
    print(f"Tiles sin contorno detectado: {len(not_detected)}")

    if len(not_detected) > 0:
        print("\nTiles sin contorno detectado:")
        for t in not_detected:
            print(
                f"Tile {t['id']:02d} | "
                f"row={t['row']}, col={t['col']} | "
                f"motivo: {t.get('contour_error')}"
            )

    return (
        kept_tiles,
        removed_tiles,
        mask_thumb_filled,
        coarse_contour_thumb,
        contour_band_thumb,
        mask_filled
    )

In [ ]:
kept_tiles, removed_tiles, mask_thumb, contour_thumb, contour_band, mask_level = \
    select_level_tiles_and_detect_background_contour(
        slide_path=r"Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs",
        target_level=4,
        tile_w=800,
        tile_h=1400,

        show_global_contour=True,
        show_all_grid=True,
        show_selected_grid=True,

        # Para ver una única imagen final por tile:
        show_tile_contours=True,
        show_tile_contour_steps=False,

        # Para ver las 5 columnas de debug por tile:
        # show_tile_contour_steps=True,

        # Para mostrar bloques resumen al final:
        show_tile_batches=False,
        batch_size=3,
    )

Final

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# JUNTAR TODOS LOS CONTORNOS DE LAS TILES EN UNA IMAGEN NEGRA
# ============================================================

def juntar_contornos_tiles_en_imagen_negra(kept_tiles, level_shape):
    """
    level_shape = mask_level.shape  -> (alto, ancho)

    Devuelve:
    - contour_global_mask: máscara booleana del contorno global
    - contour_global_img: imagen negra con el contorno en blanco
    """
    H, W = level_shape

    # máscara global vacía
    contour_global_mask = np.zeros((H, W), dtype=bool)

    for t in kept_tiles:
        # solo tiles donde sí se detectó contorno
        if not t.get("contour_detected", False):
            continue

        contour_mask = t.get("contour_mask", None)

        if contour_mask is None:
            continue

        x0 = int(t["x0"])
        y0 = int(t["y0"])
        h, w = contour_mask.shape

        # colocar el contorno local en la posición global
        contour_global_mask[y0:y0+h, x0:x0+w] |= contour_mask.astype(bool)

    # imagen negra con contorno blanco
    contour_global_img = np.zeros((H, W), dtype=np.uint8)
    contour_global_img[contour_global_mask] = 255

    return contour_global_mask, contour_global_img


# ============================================================
# EJECUCIÓN
# ============================================================

contour_global_mask, contour_global_img = juntar_contornos_tiles_en_imagen_negra(
    kept_tiles=kept_tiles,
    level_shape=mask_level.shape
)

# mostrar resultado
plt.figure(figsize=(8, 12))
plt.imshow(contour_global_img, cmap="gray")
plt.title("Contorno final unido de todas las tiles")
plt.axis("off")
plt.show()

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Si tienes contour_global_img, lo convertimos a máscara booleana
contour_mask = contour_global_img > 0

# ============================================================
# HACER EL CONTORNO MÁS VISIBLE
# ============================================================

# Grosor visual del contorno
grosor = 2   # prueba 2, 3, 4 si quieres verlo más grueso

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (2 * grosor + 1, 2 * grosor + 1)
)

contour_visible = cv2.dilate(
    contour_mask.astype(np.uint8),
    kernel,
    iterations=1
).astype(bool)

# ============================================================
# IMAGEN RGB NEGRA + CONTORNO DE COLOR
# ============================================================

H, W = contour_mask.shape

img_color = np.zeros((H, W, 3), dtype=np.uint8)

# Elige color:
# Cyan    = (0, 255, 255)
# Amarillo= (255, 255, 0)
# Magenta = (255, 0, 255)
# Verde   = (0, 255, 0)
# Rojo    = (255, 0, 0)

color_contorno = (0, 255, 255)  # cyan

img_color[contour_visible] = color_contorno

# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(8, 14), dpi=150)
plt.imshow(img_color, interpolation="nearest")
plt.title("Contorno final unido de todas las tiles")
plt.axis("off")
plt.show()